# INST447 — Week 3 Lecture Notes
## Pandas: Aggregate, Group, Join

**Course:** INST447 — Data Sources and Manipulation  
**Week:** 3  
**Lecture date:** September 17, 2026

### Learning goals
By the end of this notebook, you should be able to:

- summarize a DataFrame with scalar aggregations
- distinguish row counts from non-missing-value counts
- use `isna()` and aggregation across rows/columns
- count categories with `value_counts()`
- group records with `groupby()`
- calculate one or several summaries per group
- understand `MultiIndex` columns created by multi-aggregation
- use **named aggregation** to create readable output columns directly
- merge tables with inner, left, right, and outer joins
- understand why repeated join keys can duplicate rows
- use `validate="many_to_one"` to protect lookup-table joins
- reason carefully about missing values, assumptions, and weighting


## 1. Setup: the same flight log from Week 2

The lecture reuses the same ten-flight dataset from Week 2.  
Distance is numeric, prices are in dollars, and delays are in minutes.


In [1]:
import pandas as pd
import numpy as np

flights_data = [
    ("2024-01-15", "UA1247", "BWI", "ORD", 651,  "B737", "12A", 289.50, 15),
    ("2024-01-22", "DL456",  "ORD", "LAX", 1745, "A321", "8F",  425.00, None),
    ("2024-02-08", "WN2891", "LAX", "PHX", 370,  "B737", "",    149.99, 0),
    ("2024-02-10", "WN1055", "PHX", "DEN", 602,  "B737", "15C", None,   45),
    ("2024-03-05", "AA892",  "DEN", "DFW", 663,  "B737", "21B", 198.75, None),
    ("2024-03-12", "UA634",  "DFW", "IAD", 1216, "B777", "9A",  345.25, 12),
    ("2024-04-20", "B61840", "IAD", "BOS", 429,  "",     "11D", 179.50, 0),
    ("2024-05-15", "DL1123", "BOS", "ATL", 946,  "A220", "4A",  267.00, 25),
    ("2024-05-18", "DL2967", "ATL", "MIA", 594,  "B737", "",    None,   8),
    ("2024-06-02", "AA1456", "MIA", "LGA", 1095, "A321", "18F", 312.80, None),
]

columns = [
    "flight_date", "flight_number", "origin", "destination",
    "distance", "aircraft", "seat", "price", "delay_min"
]

flights = pd.DataFrame(flights_data, columns=columns)
flights


,flight_date,flight_number,origin,destination,distance,aircraft,seat,price,delay_min
0,2024-01-15,UA1247,BWI,ORD,651,B737,12A,289.50,15.0
1,2024-01-22,DL456,ORD,LAX,1745,A321,8F,425.00,NaN
2,2024-02-08,WN2891,LAX,PHX,370,B737,,149.99,0.0
3,2024-02-10,WN1055,PHX,DEN,602,B737,15C,NaN,45.0
4,2024-03-05,AA892,DEN,DFW,663,B737,21B,198.75,NaN
5,2024-03-12,UA634,DFW,IAD,1216,B777,9A,345.25,12.0
6,2024-04-20,B61840,IAD,BOS,429,,11D,179.50,0.0
7,2024-05-15,DL1123,BOS,ATL,946,A220,4A,267.00,25.0
8,2024-05-18,DL2967,ATL,MIA,594,B737,,NaN,8.0
9,2024-06-02,AA1456,MIA,LGA,1095,A321,18F,312.80,NaN


# Part A — Aggregation

An **aggregation** reduces many values to a smaller summary, often a single number.

Examples:

- total miles → `sum()`
- average miles → `mean()`
- longest flight → `max()`
- shortest flight → `min()`
- count of recorded values → `count()`


## 2. Summary statistics with `describe()`

In [2]:
flights.describe()

,distance,price,delay_min
count,10.000000,8.000000,7.000000
mean,831.100000,270.973750,15.000000
std,422.939568,92.249878,15.853496
min,370.000000,149.990000,0.000000
25%,596.000000,193.937500,4.000000
50%,657.000000,278.250000,12.000000
75%,1057.750000,320.912500,20.000000
max,1745.000000,425.000000,45.000000


For a mixed-type DataFrame, `describe()` normally summarizes numeric columns.

Useful rows include:

- `count`
- `mean`
- `min`
- `50%` — the median
- `max`


In [3]:
flights[["aircraft", "seat", "distance", "price", "delay_min"]].describe(include="all")

,aircraft,seat,distance,price,delay_min
count,10,10,10.000000,8.000000,7.000000
unique,5,9,NaN,NaN,NaN
top,B737,,NaN,NaN,NaN
freq,5,2,NaN,NaN,NaN
mean,NaN,NaN,831.100000,270.973750,15.000000
std,NaN,NaN,422.939568,92.249878,15.853496
min,NaN,NaN,370.000000,149.990000,0.000000
25%,NaN,NaN,596.000000,193.937500,4.000000
50%,NaN,NaN,657.000000,278.250000,12.000000
75%,NaN,NaN,1057.750000,320.912500,20.000000


For text columns, `describe(include="all")` can show:

- `unique` — number of distinct values
- `top` — a most-common value
- `freq` — frequency of that value

A blank string is still a value. It is not automatically treated as missing.


## 3. Individual scalar summaries

In [4]:
print("Flights:", len(flights))
print("Total miles:", flights.distance.sum())
print("Average miles:", flights.distance.mean())
print("Longest flight:", flights.distance.max())
print("Shortest flight:", flights.distance.min())

Flights: 10
Total miles: 8311
Average miles: 831.1
Longest flight: 1745
Shortest flight: 370


Expected lecture results:

- Flights: **10**
- Total miles: **8,311**
- Average miles: **831.1**
- Longest flight: **1,745**
- Shortest flight: **370**


## 4. `count()` vs total number of rows

In [5]:
print("Recorded prices:", flights.price.count())
print("Total rows:", len(flights))

Recorded prices: 8
Total rows: 10


`count()` skips missing values.

So:

```python
flights.price.count()
```

answers **How many non-missing prices are recorded?**

while:

```python
len(flights)
```

answers **How many flight records exist?**


## 5. Finding missing values with `isna()`

In [6]:
flights.isna()

,flight_date,flight_number,origin,destination,distance,aircraft,seat,price,delay_min
0,False,False,False,False,False,False,False,False,False
1,False,False,False,False,False,False,False,False,True
2,False,False,False,False,False,False,False,False,False
3,False,False,False,False,False,False,False,True,False
4,False,False,False,False,False,False,False,False,True
5,False,False,False,False,False,False,False,False,False
6,False,False,False,False,False,False,False,False,False
7,False,False,False,False,False,False,False,False,False
8,False,False,False,False,False,False,False,True,False
9,False,False,False,False,False,False,False,False,True


`True` means the entry is missing (`NaN` / `None`).

Important: an empty string such as `""` is **not** considered null, so `isna()` returns `False` for it.


### Missing values per column — default `axis=0`

In [7]:
flights.isna().sum()

flight_date      0
flight_number    0
origin           0
destination      0
distance         0
aircraft         0
seat             0
price            2
delay_min        3
dtype: int64

The default is `axis=0`, so pandas aggregates **down the rows** and produces one result per column.

Lecture result:

- `price`: 2 missing
- `delay_min`: 3 missing


### Missing fields per row — `axis=1`

In [8]:
flights.isna().sum(axis=1)

0    0
1    1
2    0
3    1
4    1
5    0
6    0
7    0
8    1
9    1
dtype: int64

With `axis=1`, pandas aggregates **across columns**, so each flight record receives one missing-field count.


## 6. Aggregation and `apply()`

In [9]:
a = flights.isna().sum(axis=1)
b = flights.isna().apply(sum, axis=1)

pd.DataFrame({"sum": a, "apply_sum": b})

,sum,apply_sum
0,0,0
1,1,1
2,0,0
3,1,1
4,1,1
5,0,0
6,0,0
7,0,0
8,1,1
9,1,1


These produce the same result.

Conceptually, a scalar aggregation is a summarizing operation that condenses a row or column to one value.

Do not confuse this general idea with pandas' separate `.transform()` method.


## 7. Missing data changes the denominator

In [10]:
flights.delay_min

0    15.0
1     NaN
2     0.0
3    45.0
4     NaN
5    12.0
6     0.0
7    25.0
8     8.0
9     NaN
Name: delay_min, dtype: float64

In [11]:
flights.delay_min.mean()

np.float64(15.0)

Pandas ignores missing delays when computing the mean.

There are 7 recorded delay values totaling 105 minutes:

\[
105 / 7 = 15
\]

So the mean recorded delay is **15 minutes**.


### Filling missing delays with zero changes the assumption

In [12]:
flights.delay_min.fillna(0).mean()

np.float64(10.5)

The new mean is **10.5 minutes** because there are now 10 values.

This is not automatically a better answer. Filling missing delay values with zero assumes the unrecorded flights had **zero delay**.


# Part B — A data problem pandas cannot solve by itself

The lecture uses seat labels to show an important boundary: character patterns do not necessarily tell us physical meaning.

For one illustrative 3–3 layout:

```text
window  A B C   aisle   D E F  window
```

A simple pattern can recognize seats ending in `A` or `F`.


In [13]:
pd.Series(["12A", "8F", "15C", "21B"]).str.fullmatch(r"[0-9]+[AF]")

0     True
1     True
2    False
3    False
dtype: bool

But another aircraft/cabin layout could assign different physical meanings to the same seat letter.

For example, a `B` seat might be a middle seat in one layout but an aisle seat in another.

### Lesson
A regex recognizes text patterns. It does **not** know the actual seat map.

To classify a seat reliably, the workflow may need external information such as:

- airline
- aircraft configuration
- flight/date
- a seat-map knowledge base or API

The lecture also discusses how an LLM API could participate in the workflow, but external lookup/tool access is still required when the needed information is not already supplied.


### Decomposing the seat-preference question

To answer “Do I prefer window, middle, or aisle seats?” you would need to:

1. identify the layout for each flight
2. classify each recorded seat
3. handle unknown layouts and missing seats
4. count seat categories
5. interpret whether those observed seats actually indicate preference

Even perfect counts do not prove that the passenger freely chose each seat.


# Part C — Categories and grouping


## 8. Extract the airline code

In [14]:
flights["airline_code"] = flights.flight_number.str[:2]
flights[["flight_number", "airline_code"]]

,flight_number,airline_code
0,UA1247,UA
1,DL456,DL
2,WN2891,WN
3,WN1055,WN
4,AA892,AA
5,UA634,UA
6,B61840,B6
7,DL1123,DL
8,DL2967,DL
9,AA1456,AA


`.str[:2]` slices the first two characters of **every string** in the Series.

This is different from:

```python
flights.flight_number[:2]
```

which would select the first two rows.


## 9. Unique values

In [15]:
flights.airline_code.unique()

array(['UA', 'DL', 'WN', 'AA', 'B6'], dtype=object)

In [16]:
flights.airline_code.unique().size

5

The lecture flight log contains **5** airline codes.

## 10. Count categories with `value_counts()`

In [17]:
flights.airline_code.value_counts()

airline_code
DL    3
UA    2
WN    2
AA    2
B6    1
Name: count, dtype: int64

Lecture result:

- DL: 3 flights
- UA: 2
- WN: 2
- AA: 2
- B6: 1

`value_counts()` is useful when the question is simply **How often does each category occur?**


# Part D — `groupby()`

`groupby()` organizes rows into groups so a calculation can be performed separately for each group.

A useful mental model is:

**split → apply calculation → combine results**


## 11. Create a GroupBy object

In [18]:
airline_groups = flights.groupby("airline_code")
airline_groups

This is not yet a summary table. It is a GroupBy object waiting for a calculation.


## 12. Count rows in each group

In [19]:
flights.groupby("airline_code").size()

airline_code
AA    2
B6    1
DL    3
UA    2
WN    2
dtype: int64

`.size()` counts **rows** in each group.

This produces the same airline frequencies as `value_counts()`, though the ordering can differ.


## 13. Total distance by airline

In [20]:
flights.groupby("airline_code").distance.sum()

airline_code
AA    1758
B6     429
DL    3285
UA    1867
WN     972
Name: distance, dtype: int64

Read this left to right:

1. group rows by `airline_code`
2. select `distance`
3. sum distance within each airline


## 14. Maximum distance by airline

In [21]:
flight_distances = flights.groupby("airline_code").distance.max()
flight_distances

airline_code
AA    1095
B6     429
DL    1745
UA    1216
WN     602
Name: distance, dtype: int64

This gives the maximum **distance value** for each airline. It does not automatically return the complete row of the longest flight.


### Turn the grouped Series into a regular table

In [22]:
flight_distances = flight_distances.reset_index()
flight_distances

,airline_code,distance
0,AA,1095
1,B6,429
2,DL,1745
3,UA,1216
4,WN,602


`reset_index()` moves the airline labels out of the index and into a normal column.

## 15. `.size()` vs `.count()` within groups

In [23]:
flights.groupby("airline_code").size()

airline_code
AA    2
B6    1
DL    3
UA    2
WN    2
dtype: int64

In [24]:
flights.groupby("airline_code").price.count()

airline_code
AA    2
B6    1
DL    2
UA    2
WN    1
Name: price, dtype: int64

These answer different questions:

- `.size()` → number of rows/flights
- `.price.count()` → number of **non-missing prices**

Southwest has 2 flights, but only 1 recorded price.


## 16. Average delay by airline

In [25]:
flights.groupby("airline_code").delay_min.mean()

airline_code
AA     NaN
B6     0.0
DL    16.5
UA    13.5
WN    22.5
Name: delay_min, dtype: float64

Notice the difference between:

- **B6 = 0.0** → a recorded zero delay exists
- **AA = NaN** → neither AA flight has a recorded delay

Zero and missing are not the same.


# Part E — Several aggregations at once


## 17. Multi-aggregation with `.agg()`

In [26]:
airline_stats = flights.groupby("airline_code").agg({
    "distance": ["count", "sum", "mean", "max"],
    "price": ["mean", "min", "max"],
    "delay_min": "mean"
})
airline_stats

distance                        price                 delay_min
                count   sum    mean   max     mean     min     max      mean
airline_code                                                                
AA                  2  1758   879.0  1095  255.775  198.75  312.80       NaN
B6                  1   429   429.0   429  179.500  179.50  179.50       0.0
DL                  3  3285  1095.0  1745  346.000  267.00  425.00      16.5
UA                  2  1867   933.5  1216  317.375  289.50  345.25      13.5
WN                  2   972   486.0   602  149.990  149.99  149.99      22.5

The result has **two-level column names**:

- original input column
- aggregation function

For example:

```text
("distance", "sum")
```


### Inspect the column names

In [27]:
airline_stats.columns

MultiIndex([( 'distance', 'count'),
            ( 'distance',   'sum'),
            ( 'distance',  'mean'),
            ( 'distance',   'max'),
            (    'price',  'mean'),
            (    'price',   'min'),
            (    'price',   'max'),
            ('delay_min',  'mean')],
           )

### `reset_index()` does not flatten MultiIndex columns

In [28]:
airline_stats.reset_index().columns

MultiIndex([('airline_code',      ''),
            (    'distance', 'count'),
            (    'distance',   'sum'),
            (    'distance',  'mean'),
            (    'distance',   'max'),
            (       'price',  'mean'),
            (       'price',   'min'),
            (       'price',   'max'),
            (   'delay_min',  'mean')],
           )

Resetting the row index and flattening column names are different operations.


## 18. Flatten the MultiIndex column names

In [29]:
airline_stats.columns = [
    "_".join(col).strip()
    for col in airline_stats.columns
]

airline_stats.columns

Index(['distance_count', 'distance_sum', 'distance_mean', 'distance_max',
       'price_mean', 'price_min', 'price_max', 'delay_min_mean'],
      dtype='object')

Now `(distance, sum)` becomes `distance_sum`.

### Put the airline code into a column

In [30]:
airline_stats = airline_stats.reset_index()
airline_stats

,airline_code,distance_count,distance_sum,distance_mean,distance_max,price_mean,price_min,price_max,delay_min_mean
0,AA,2,1758,879.0,1095,255.775,198.75,312.80,NaN
1,B6,1,429,429.0,429,179.500,179.50,179.50,0.0
2,DL,3,3285,1095.0,1745,346.000,267.00,425.00,16.5
3,UA,2,1867,933.5,1216,317.375,289.50,345.25,13.5
4,WN,2,972,486.0,602,149.990,149.99,149.99,22.5


### Rename to readable summary names

In [31]:
airline_stats.columns = [
    "airline_code",
    "flight_count",
    "total_distance",
    "avg_distance",
    "max_distance",
    "avg_price",
    "min_price",
    "max_price",
    "avg_delay"
]

airline_stats

,airline_code,flight_count,total_distance,avg_distance,max_distance,avg_price,min_price,max_price,avg_delay
0,AA,2,1758,879.0,1095,255.775,198.75,312.80,NaN
1,B6,1,429,429.0,429,179.500,179.50,179.50,0.0
2,DL,3,3285,1095.0,1745,346.000,267.00,425.00,16.5
3,UA,2,1867,933.5,1216,317.375,289.50,345.25,13.5
4,WN,2,972,486.0,602,149.990,149.99,149.99,22.5


# Part F — Named aggregation

The lecture introduces a shorter pandas syntax called **named aggregation**.

It lets you specify the output column name while also specifying:

1. the input column
2. the aggregation function


In [32]:
airline_stats_named = (
    flights.groupby("airline_code")
    .agg(
        flight_count=("flight_number", "size"),
        total_distance=("distance", "sum"),
        avg_distance=("distance", "mean"),
        max_distance=("distance", "max"),
        avg_price=("price", "mean"),
        min_price=("price", "min"),
        max_price=("price", "max"),
        avg_delay=("delay_min", "mean"),
    )
    .reset_index()
)

airline_stats_named

,airline_code,flight_count,total_distance,avg_distance,max_distance,avg_price,min_price,max_price,avg_delay
0,AA,2,1758,879.0,1095,255.775,198.75,312.80,NaN
1,B6,1,429,429.0,429,179.500,179.50,179.50,0.0
2,DL,3,3285,1095.0,1745,346.000,267.00,425.00,16.5
3,UA,2,1867,933.5,1216,317.375,289.50,345.25,13.5
4,WN,2,972,486.0,602,149.990,149.99,149.99,22.5


Example:

```python
total_distance=("distance", "sum")
```

means:

- output column name → `total_distance`
- input column → `distance`
- calculation → `sum`


## 19. Group by more than one column

In [33]:
multi_group = (
    flights.groupby(["airline_code", "origin"])
    .size()
    .reset_index(name="flight_count")
)

multi_group

,airline_code,origin,flight_count
0,AA,DEN,1
1,AA,MIA,1
2,B6,IAD,1
3,DL,ATL,1
4,DL,BOS,1
5,DL,ORD,1
6,UA,BWI,1
7,UA,DFW,1
8,WN,LAX,1
9,WN,PHX,1


The group is defined by the **combination** of both columns.


In [34]:
(
    flights.groupby(["airline_code", "aircraft"])
    .size()
    .reset_index(name="flight_count")
)

,airline_code,aircraft,flight_count
0,AA,A321,1
1,AA,B737,1
2,B6,,1
3,DL,A220,1
4,DL,A321,1
5,DL,B737,1
6,UA,B737,1
7,UA,B777,1
8,WN,B737,2


Changing the grouping columns changes the question being asked.  
In the lecture data, the WN/B737 combination appears twice.


# Part G — Joining tables

A join/merge attaches information from another table using matching key values.

The flight table has airline codes. A lookup table can translate those codes into airline names.


In [35]:
airlines = pd.DataFrame(
    [
        ("UA", "United Airlines"),
        ("DL", "Delta Air Lines"),
        ("WN", "Southwest Airlines"),
        ("AA", "American Airlines"),
        ("AS", "Alaska Airlines"),
    ],
    columns=["code", "airline_name"]
)

airlines

,code,airline_name
0,UA,United Airlines
1,DL,Delta Air Lines
2,WN,Southwest Airlines
3,AA,American Airlines
4,AS,Alaska Airlines


Notice that the lookup includes `AS` but not `B6`. This gives us useful unmatched-key examples.


## 20. Inner join

In [36]:
flights_inner = flights.merge(
    airlines,
    left_on="airline_code",
    right_on="code",
    how="inner"
)

flights_inner

,flight_date,flight_number,origin,destination,distance,aircraft,seat,price,delay_min,airline_code,code,airline_name
0,2024-01-15,UA1247,BWI,ORD,651,B737,12A,289.50,15.0,UA,UA,United Airlines
1,2024-01-22,DL456,ORD,LAX,1745,A321,8F,425.00,NaN,DL,DL,Delta Air Lines
2,2024-02-08,WN2891,LAX,PHX,370,B737,,149.99,0.0,WN,WN,Southwest Airlines
3,2024-02-10,WN1055,PHX,DEN,602,B737,15C,NaN,45.0,WN,WN,Southwest Airlines
4,2024-03-05,AA892,DEN,DFW,663,B737,21B,198.75,NaN,AA,AA,American Airlines
5,2024-03-12,UA634,DFW,IAD,1216,B777,9A,345.25,12.0,UA,UA,United Airlines
6,2024-05-15,DL1123,BOS,ATL,946,A220,4A,267.00,25.0,DL,DL,Delta Air Lines
7,2024-05-18,DL2967,ATL,MIA,594,B737,,NaN,8.0,DL,DL,Delta Air Lines
8,2024-06-02,AA1456,MIA,LGA,1095,A321,18F,312.80,NaN,AA,AA,American Airlines


In [37]:
print("Original rows:", len(flights))
print("Inner-join rows:", len(flights_inner))

Original rows: 10
Inner-join rows: 9


An **inner join** keeps only matching keys.

Because `B6` has no matching lookup row, that flight disappears from the inner-join result.


## 21. Left join

In [38]:
flights_left = flights.merge(
    airlines,
    left_on="airline_code",
    right_on="code",
    how="left"
)

flights_left

,flight_date,flight_number,origin,destination,distance,aircraft,seat,price,delay_min,airline_code,code,airline_name
0,2024-01-15,UA1247,BWI,ORD,651,B737,12A,289.50,15.0,UA,UA,United Airlines
1,2024-01-22,DL456,ORD,LAX,1745,A321,8F,425.00,NaN,DL,DL,Delta Air Lines
2,2024-02-08,WN2891,LAX,PHX,370,B737,,149.99,0.0,WN,WN,Southwest Airlines
3,2024-02-10,WN1055,PHX,DEN,602,B737,15C,NaN,45.0,WN,WN,Southwest Airlines
4,2024-03-05,AA892,DEN,DFW,663,B737,21B,198.75,NaN,AA,AA,American Airlines
5,2024-03-12,UA634,DFW,IAD,1216,B777,9A,345.25,12.0,UA,UA,United Airlines
6,2024-04-20,B61840,IAD,BOS,429,,11D,179.50,0.0,B6,NaN,NaN
7,2024-05-15,DL1123,BOS,ATL,946,A220,4A,267.00,25.0,DL,DL,Delta Air Lines
8,2024-05-18,DL2967,ATL,MIA,594,B737,,NaN,8.0,DL,DL,Delta Air Lines
9,2024-06-02,AA1456,MIA,LGA,1095,A321,18F,312.80,NaN,AA,AA,American Airlines


A **left join** keeps every row from the left table (`flights`).

Unmatched lookup information becomes missing.


## 22. Right join

In [39]:
flights_right = flights.merge(
    airlines,
    left_on="airline_code",
    right_on="code",
    how="right"
)

flights_right

,flight_date,flight_number,origin,destination,distance,aircraft,seat,price,delay_min,airline_code,code,airline_name
0,2024-01-15,UA1247,BWI,ORD,651.0,B737,12A,289.50,15.0,UA,UA,United Airlines
1,2024-03-12,UA634,DFW,IAD,1216.0,B777,9A,345.25,12.0,UA,UA,United Airlines
2,2024-01-22,DL456,ORD,LAX,1745.0,A321,8F,425.00,NaN,DL,DL,Delta Air Lines
3,2024-05-15,DL1123,BOS,ATL,946.0,A220,4A,267.00,25.0,DL,DL,Delta Air Lines
4,2024-05-18,DL2967,ATL,MIA,594.0,B737,,NaN,8.0,DL,DL,Delta Air Lines
5,2024-02-08,WN2891,LAX,PHX,370.0,B737,,149.99,0.0,WN,WN,Southwest Airlines
6,2024-02-10,WN1055,PHX,DEN,602.0,B737,15C,NaN,45.0,WN,WN,Southwest Airlines
7,2024-03-05,AA892,DEN,DFW,663.0,B737,21B,198.75,NaN,AA,AA,American Airlines
8,2024-06-02,AA1456,MIA,LGA,1095.0,A321,18F,312.80,NaN,AA,AA,American Airlines
9,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,AS,Alaska Airlines


A **right join** keeps every row/key from the right table (`airlines`).

That means the unmatched `AS` lookup entry is retained, with missing flight fields.


## 23. Outer join

In [40]:
flights_outer = flights.merge(
    airlines,
    left_on="airline_code",
    right_on="code",
    how="outer"
)

flights_outer

,flight_date,flight_number,origin,destination,distance,aircraft,seat,price,delay_min,airline_code,code,airline_name
0,2024-03-05,AA892,DEN,DFW,663.0,B737,21B,198.75,NaN,AA,AA,American Airlines
1,2024-06-02,AA1456,MIA,LGA,1095.0,A321,18F,312.80,NaN,AA,AA,American Airlines
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,AS,Alaska Airlines
3,2024-04-20,B61840,IAD,BOS,429.0,,11D,179.50,0.0,B6,NaN,NaN
4,2024-01-22,DL456,ORD,LAX,1745.0,A321,8F,425.00,NaN,DL,DL,Delta Air Lines
5,2024-05-15,DL1123,BOS,ATL,946.0,A220,4A,267.00,25.0,DL,DL,Delta Air Lines
6,2024-05-18,DL2967,ATL,MIA,594.0,B737,,NaN,8.0,DL,DL,Delta Air Lines
7,2024-01-15,UA1247,BWI,ORD,651.0,B737,12A,289.50,15.0,UA,UA,United Airlines
8,2024-03-12,UA634,DFW,IAD,1216.0,B777,9A,345.25,12.0,UA,UA,United Airlines
9,2024-02-08,WN2891,LAX,PHX,370.0,B737,,149.99,0.0,WN,WN,Southwest Airlines


An **outer join** keeps unmatched keys from both tables.

### Join summary

| Join | Keys retained |
|---|---|
| `inner` | matches only |
| `left` | all left keys + matches |
| `right` | all right keys + matches |
| `outer` | all keys from both |


# Part H — Repeated join keys

A join is based on key matches, not row positions.

If a lookup table accidentally has more than one row for a key, one source record may match several lookup rows.


In [41]:
airlines_duplicate = airlines.copy()
airlines_duplicate.loc[len(airlines_duplicate)] = ["UA", "United Airlines"]
airlines_duplicate

,code,airline_name
0,UA,United Airlines
1,DL,Delta Air Lines
2,WN,Southwest Airlines
3,AA,American Airlines
4,AS,Alaska Airlines
5,UA,United Airlines


In [42]:
flights_duplicate = flights.merge(
    airlines_duplicate,
    left_on="airline_code",
    right_on="code",
    how="left"
)

flights_duplicate

,flight_date,flight_number,origin,destination,distance,aircraft,seat,price,delay_min,airline_code,code,airline_name
0,2024-01-15,UA1247,BWI,ORD,651,B737,12A,289.50,15.0,UA,UA,United Airlines
1,2024-01-15,UA1247,BWI,ORD,651,B737,12A,289.50,15.0,UA,UA,United Airlines
2,2024-01-22,DL456,ORD,LAX,1745,A321,8F,425.00,NaN,DL,DL,Delta Air Lines
3,2024-02-08,WN2891,LAX,PHX,370,B737,,149.99,0.0,WN,WN,Southwest Airlines
4,2024-02-10,WN1055,PHX,DEN,602,B737,15C,NaN,45.0,WN,WN,Southwest Airlines
5,2024-03-05,AA892,DEN,DFW,663,B737,21B,198.75,NaN,AA,AA,American Airlines
6,2024-03-12,UA634,DFW,IAD,1216,B777,9A,345.25,12.0,UA,UA,United Airlines
7,2024-03-12,UA634,DFW,IAD,1216,B777,9A,345.25,12.0,UA,UA,United Airlines
8,2024-04-20,B61840,IAD,BOS,429,,11D,179.50,0.0,B6,NaN,NaN
9,2024-05-15,DL1123,BOS,ATL,946,A220,4A,267.00,25.0,DL,DL,Delta Air Lines


In [43]:
print("Original rows:", len(flights))
print("Rows after merge:", len(flights_duplicate))

Original rows: 10
Rows after merge: 12


There are 2 UA flights and now 2 UA lookup entries.

Each UA flight matches both lookup entries, so the result expands from 10 rows to 12.


## 24. Protect a lookup join with `validate='many_to_one'`

In [44]:
try:
    flights.merge(
        airlines_duplicate,
        left_on="airline_code",
        right_on="code",
        how="left",
        validate="many_to_one"
    )
except Exception as e:
    print(type(e).__name__ + ":", e)

MergeError: Merge keys are not unique in right dataset; not a many-to-one merge


`many_to_one` means:

- many flight rows may share the same airline code
- the lookup table should have **at most one row per key**

If the right-side lookup contains duplicate keys, pandas stops the merge with an error instead of silently accepting duplicated output.


### The original lookup passes validation

In [45]:
flights_checked = flights.merge(
    airlines,
    left_on="airline_code",
    right_on="code",
    how="left",
    validate="many_to_one"
)

flights_checked

,flight_date,flight_number,origin,destination,distance,aircraft,seat,price,delay_min,airline_code,code,airline_name
0,2024-01-15,UA1247,BWI,ORD,651,B737,12A,289.50,15.0,UA,UA,United Airlines
1,2024-01-22,DL456,ORD,LAX,1745,A321,8F,425.00,NaN,DL,DL,Delta Air Lines
2,2024-02-08,WN2891,LAX,PHX,370,B737,,149.99,0.0,WN,WN,Southwest Airlines
3,2024-02-10,WN1055,PHX,DEN,602,B737,15C,NaN,45.0,WN,WN,Southwest Airlines
4,2024-03-05,AA892,DEN,DFW,663,B737,21B,198.75,NaN,AA,AA,American Airlines
5,2024-03-12,UA634,DFW,IAD,1216,B777,9A,345.25,12.0,UA,UA,United Airlines
6,2024-04-20,B61840,IAD,BOS,429,,11D,179.50,0.0,B6,NaN,NaN
7,2024-05-15,DL1123,BOS,ATL,946,A220,4A,267.00,25.0,DL,DL,Delta Air Lines
8,2024-05-18,DL2967,ATL,MIA,594,B737,,NaN,8.0,DL,DL,Delta Air Lines
9,2024-06-02,AA1456,MIA,LGA,1095,A321,18F,312.80,NaN,AA,AA,American Airlines


# Part I — Column-name conflicts in chained merges


In [46]:
airport_info = pd.DataFrame({
    "code": ["BWI", "ORD"],
    "city": ["Baltimore", "Chicago"],
    "distance": [25, 18]
})

airport_info

,code,city,distance
0,BWI,Baltimore,25
1,ORD,Chicago,18


This is a toy table. Here `distance` means distance from the airport to the city center, not flight distance.


In [47]:
flights_airports = (
    flights
    .merge(
        airlines,
        left_on="airline_code",
        right_on="code",
        how="inner"
    )
    .merge(
        airport_info,
        left_on="origin",
        right_on="code",
        how="inner"
    )
)

flights_airports

,flight_date,flight_number,origin,destination,distance_x,aircraft,seat,price,delay_min,airline_code,code_x,airline_name,code_y,city,distance_y
0,2024-01-15,UA1247,BWI,ORD,651,B737,12A,289.5,15.0,UA,UA,United Airlines,BWI,Baltimore,25
1,2024-01-22,DL456,ORD,LAX,1745,A321,8F,425.0,NaN,DL,DL,Delta Air Lines,ORD,Chicago,18


Both tables contain columns named `distance` and/or `code`, so pandas creates suffixes such as:

- `distance_x`
- `distance_y`
- `code_x`
- `code_y`

The suffixes prevent overlapping column names from overwriting each other.


# Part J — Interpreting summaries carefully


## 25. Average distance per airline

In [48]:
distance_summary = flights.groupby("airline_code").agg(
    flights=("flight_number", "size"),
    total_miles=("distance", "sum"),
    mean_miles=("distance", "mean")
)

distance_summary

,flights,total_miles,mean_miles
airline_code,,,
AA,2,1758,879.0
B6,1,429,429.0
DL,3,3285,1095.0
UA,2,1867,933.5
WN,2,972,486.0


## 26. Average the airline averages vs average all flights

In [49]:
print("Mean of airline means:", distance_summary.mean_miles.mean())
print("Mean across flights:", flights.distance.mean())

Mean of airline means: 764.5
Mean across flights: 831.1


Lecture results:

- mean of airline means → **764.5**
- mean across all flights → **831.1**

Why are they different?

Because the first calculation gives each **airline** equal weight.  
The second calculation gives each **flight** equal weight.

Delta contributes three flights while B6 contributes one, but averaging airline means gives Delta and B6 the same weight.

Neither expression is a coding error. They answer different questions.


## 27. Main data-quality lesson

Pandas can correctly calculate exactly what you ask for while the interpretation can still depend on analytical choices.

Examples from this lecture:

- filling missing delays with zero changes an assumption
- using `count()` instead of `size()` changes what is counted
- selecting grouping columns changes the question
- averaging group means changes weighting
- seat counts still do not prove seat preference
- join type determines which unmatched records are retained
- duplicate lookup keys can expand the dataset

Always ask:

> **What question does this calculation actually answer?**


# Week 3 Cheat Sheet

```python
# Summary
df.describe()
df["x"].sum()
df["x"].mean()
df["x"].min()
df["x"].max()
df["x"].count()

# Missing values
df.isna()
df.isna().sum()          # missing per column
df.isna().sum(axis=1)    # missing per row
df["x"].fillna(0)

# Categories
df["category"].unique()
df["category"].unique().size
df["category"].value_counts()

# Grouping
df.groupby("group").size()
df.groupby("group")["x"].sum()
df.groupby("group")["x"].mean()

# Multiple groups
df.groupby(["group1", "group2"]).size()

# Multiple aggregations
df.groupby("group").agg({
    "x": ["sum", "mean", "max"],
    "y": ["mean", "min"]
})

# Named aggregation
df.groupby("group").agg(
    count=("id", "size"),
    total=("x", "sum"),
    average=("x", "mean")
)

# Turn index into column
result.reset_index()

# Merge
left.merge(
    right,
    left_on="left_key",
    right_on="right_key",
    how="left"
)

# Join validation
left.merge(
    right,
    left_on="key",
    right_on="key",
    how="left",
    validate="many_to_one"
)
```


# Practice

Try these before opening the solution section.

### 1.
How many non-missing delay values are recorded?

### 2.
Calculate total distance by airline.

### 3.
Create a table containing each airline's:
- number of flights
- average price
- average delay

Use named aggregation.

### 4.
Which is safer for attaching airline names to the full flight log: an inner join or left join, if you cannot assume every flight code exists in the lookup? Explain what each does.

### 5.
Why can a duplicate key in a lookup table increase the number of rows after a merge?

### 6.
Calculate missing fields per flight record.

### 7.
Compare the mean of airline mean distances with the overall flight mean. Explain why they differ.


# Practice Solutions

In [50]:
# 1
flights.delay_min.count()

np.int64(7)

In [51]:
# 2
flights.groupby("airline_code").distance.sum()

airline_code
AA    1758
B6     429
DL    3285
UA    1867
WN     972
Name: distance, dtype: int64

In [52]:
# 3
practice_summary = (
    flights.groupby("airline_code")
    .agg(
        flight_count=("flight_number", "size"),
        avg_price=("price", "mean"),
        avg_delay=("delay_min", "mean"),
    )
    .reset_index()
)

practice_summary

,airline_code,flight_count,avg_price,avg_delay
0,AA,2,255.775,NaN
1,B6,1,179.500,0.0
2,DL,3,346.000,16.5
3,UA,2,317.375,13.5
4,WN,2,149.990,22.5


### 4 — Explanation

A **left join** retains every flight from the flight log and fills unmatched airline information with missing values.

An **inner join** drops flights whose airline code has no match in the lookup.

The appropriate choice depends on the question, but if the goal is specifically to preserve the complete flight log, the left join does that.


### 5 — Explanation

If the lookup has multiple rows with the same key, a source row can match each of them.

For example:

- 2 UA flight rows
- 2 UA lookup rows

produce:

- 2 × 2 = 4 UA rows in the merged result

instead of 2.


In [53]:
# 6
flights.isna().sum(axis=1)

0    0
1    1
2    0
3    1
4    1
5    0
6    0
7    0
8    1
9    1
dtype: int64

In [54]:
# 7
mean_of_airline_means = distance_summary.mean_miles.mean()
overall_flight_mean = flights.distance.mean()

mean_of_airline_means, overall_flight_mean

(np.float64(764.5), np.float64(831.1))

The values differ because the first gives every airline one equal contribution, while the second gives every flight one equal contribution.


# Final Takeaways

1. **Aggregation** summarizes many values.
2. Missing values matter because functions such as `count()` and `mean()` usually skip them.
3. `axis=0` and `axis=1` determine the direction of an operation.
4. `value_counts()` is ideal for simple category frequencies.
5. `groupby()` lets you calculate summaries separately for categories.
6. `.size()` counts rows; `.count()` counts non-missing values.
7. `.agg()` can calculate several statistics at once.
8. Named aggregation is often the cleanest way to build readable grouped summaries.
9. `merge()` joins tables by matching key values, not by row position.
10. Join type determines which unmatched records survive.
11. Duplicate lookup keys can duplicate source rows; `validate="many_to_one"` can catch that problem.
12. A correct pandas calculation can still answer the wrong question if assumptions, weighting, missingness, or grouping choices are not considered.
